
# AsyncIO — учебный ноутбук (Python)

Этот ноутбук — **пошаговый и запускаемый гайд** по асинхронному программированию в Python с использованием `asyncio`.

⚠️ **Важно для Jupyter**
- В Jupyter уже запущен event loop.
- Поэтому **не используем `asyncio.run(...)`**.
- Вместо этого в ячейках пишем просто: `await some_async_func()`.

Версия Python: рекомендуется **3.11+** (для `TaskGroup`).



## 1. Корутины и `await`

- `async def` объявляет **корутину**
- Вызов корутины **не запускает её**, а возвращает coroutine object
- `await` — точка, где управление возвращается event loop


In [2]:

import asyncio

async def hello():
    print("Hello...")
    await asyncio.sleep(1)
    print("...world!")

# В Jupyter просто await
await hello()


Hello...
...world!



## 2. `time.sleep` vs `await asyncio.sleep`

- `time.sleep` **блокирует** event loop
- `await asyncio.sleep` — **неблокирующая пауза**


In [3]:

import time
import asyncio

async def bad():
    time.sleep(1)
    return "bad done"

async def good():
    await asyncio.sleep(1)
    return "good done"

start = time.perf_counter()
await asyncio.gather(bad(), bad())
print("bad:", time.perf_counter() - start)

start = time.perf_counter()
await asyncio.gather(good(), good())
print("good:", time.perf_counter() - start)


bad: 2.00039877800009
good: 1.001017057998979



## 3. Конкурентность: `asyncio.gather`


In [ ]:

async def work(i, delay):
    await asyncio.sleep(delay)
    return f"task {i}"

results = await asyncio.gather(
    work(1, 1),
    work(2, 0.5),
    work(3, 0.2),
)

results



## 4. `asyncio.create_task` и `Task`

- `Task` — запланированная корутина
- Выполняется **параллельно**, даже если мы её не await-им сразу


In [ ]:

async def demo_task():
    await asyncio.sleep(1)
    print("task finished")

task = asyncio.create_task(demo_task())
print("created task")
await asyncio.sleep(0.2)
print("still running...")
await task



## 5. Отмена задач (`cancel`) и `CancelledError`


In [ ]:

async def cancellable():
    try:
        while True:
            print("working...")
            await asyncio.sleep(0.3)
    except asyncio.CancelledError:
        print("cancelled!")
        raise
    finally:
        print("cleanup")

task = asyncio.create_task(cancellable())
await asyncio.sleep(1)
task.cancel()

try:
    await task
except asyncio.CancelledError:
    print("caught CancelledError")



## 6. Таймауты: `asyncio.wait_for`


In [ ]:

async def slow():
    await asyncio.sleep(2)
    return "done"

try:
    await asyncio.wait_for(slow(), timeout=1)
except asyncio.TimeoutError:
    print("timeout!")



## 7. TaskGroup (Python 3.11+)


In [ ]:

import sys
import asyncio

if sys.version_info >= (3, 11):
    async def tg_demo():
        async with asyncio.TaskGroup() as tg:
            tg.create_task(asyncio.sleep(0.5))
            tg.create_task(asyncio.sleep(1))
        print("all tasks done")

    await tg_demo()
else:
    print("TaskGroup недоступен — пропускаем")



## 8. Async I/O без интернета: TCP echo server


In [ ]:

async def handle_echo(reader, writer):
    data = await reader.read(100)
    writer.write(data)
    await writer.drain()
    writer.close()

server = await asyncio.start_server(handle_echo, "127.0.0.1", 0)
addr = server.sockets[0].getsockname()

async def client(msg):
    reader, writer = await asyncio.open_connection(*addr)
    writer.write(msg.encode())
    await writer.drain()
    data = await reader.read(100)
    writer.close()
    return data.decode()

result = await client("hello")
server.close()
await server.wait_closed()
result



## 9. Producer / Consumer: `asyncio.Queue`


In [ ]:

async def producer(q):
    for i in range(5):
        await q.put(i)
    await q.put(None)

async def consumer(q):
    while True:
        item = await q.get()
        if item is None:
            break
        print("consumed", item)

q = asyncio.Queue()
await asyncio.gather(producer(q), consumer(q))



## 10. Ограничение конкурентности: `Semaphore`


In [ ]:

sem = asyncio.Semaphore(2)

async def limited(i):
    async with sem:
        print("start", i)
        await asyncio.sleep(1)
        print("end", i)

await asyncio.gather(*(limited(i) for i in range(5)))



## 11. Блокирующий код: `asyncio.to_thread`


In [ ]:

import time

def blocking(x):
    time.sleep(1)
    return x * x

async def run():
    return await asyncio.gather(
        asyncio.to_thread(blocking, 2),
        asyncio.to_thread(blocking, 3),
    )

await run()



## 12. Async generators


In [ ]:

async def agen():
    for i in range(3):
        await asyncio.sleep(0.5)
        yield i

async for x in agen():
    print(x)



## 13. Async context manager


In [ ]:

class AsyncResource:
    async def __aenter__(self):
        print("enter")
        return self

    async def __aexit__(self, exc_type, exc, tb):
        print("exit")

async with AsyncResource():
    print("inside")



## Шпаргалка

- `async def` — корутина
- `await` — отдаёт управление event loop
- `gather` — ждать несколько задач
- `create_task` — запланировать выполнение
- `Semaphore` — лимит конкурентности
- `Queue` — producer/consumer
- `to_thread` — CPU / blocking код
